<a href="https://colab.research.google.com/github/dipeshMahakali/reinforcement_model/blob/main/PytorchPrac1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch


In [2]:
x = torch.tensor(3.0, requires_grad=True)  #stands for Requires Gradient(pytorch will know now we will calculate its derivative)

In [3]:
y = x**2

In [4]:
print(x,y) # grad_fn stores what operation we have perform, so it can use at derivation time

tensor(3., requires_grad=True) tensor(9., grad_fn=<PowBackward0>)


In [5]:
y.backward() # it is pytorch function to calculate derivative of y

In [6]:
print(x.grad) # it will show derivartive value

tensor(6.)


In [7]:
x = torch.tensor(3.0, requires_grad=True)

In [8]:
y = x ** 2

In [9]:
z = torch.sin(y)

In [10]:
print(x,y,z)

tensor(3., requires_grad=True) tensor(9., grad_fn=<PowBackward0>) tensor(0.4121, grad_fn=<SinBackward0>)


In [11]:
z.backward()

In [12]:
print(x.grad)

tensor(-5.4668)


In [13]:
# Traning small neural network to predict student will get placement according to his cgpa
x = torch.tensor(6.9) #input
y = torch.tensor(0.0) #true label

w = torch.tensor(1.0) #weight
b = torch.tensor(0.0) #bias

In [14]:
# binary cross entropy loss for scalar
def binary_cross_entropy_loss(prediction, target):
  epsilon = 1e-8 # to prevent log(0)
  prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
  return -(target*torch.log(prediction)+(1-target)*torch.log(1-prediction))

In [16]:
# Forward Pass
z=w*x+b  # weighted sum (linear part)[Linear Transformation equation]
y_pred = torch.sigmoid(z) # predicted probability [Activation (sigmoid function)]

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)
print(loss)

tensor(6.9010)


In [17]:
# Derivatives:
# 1: dL/d(y_pred) Loss with respect to prediction(y_pred)
d_loss_dy_pred =  (y_pred-y)/(y_pred*(1-y_pred))

# 2: dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid prediction)
dy_pred_dz = y_pred * (1-y_pred)

# 3: dz/dw and dz/db: z with respect to w and b
dz_dw = x # dz/dw = x
dz_db = 1 # dz/db = 1 (bias contributes directly to z)

dL_dw = d_loss_dy_pred * dy_pred_dz * dz_dw
dL_db = d_loss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of Loss w.r.t weight(dw): {dL_dw}")
print(f"Manual Gradient of Loss w.r.t bais(db): {dL_db}")

Manual Gradient of Loss w.r.t weight(dw): 6.8930535316467285
Manual Gradient of Loss w.r.t bais(db): 0.9989932179450989


In [18]:
# Above was manual way to calculate loss and derivative now we will use autograd to calculate same things

In [19]:
x = torch.tensor(6.9)

In [20]:
y = torch.tensor(0)

In [21]:
w = torch.tensor(1.0, requires_grad=True)

b = torch.tensor(0.0, requires_grad=True)

In [22]:
z = w * x + b

In [23]:
y_pred = torch.sigmoid(z)

In [24]:
print(z,y_pred)

tensor(6.9000, grad_fn=<AddBackward0>) tensor(0.9990, grad_fn=<SigmoidBackward0>)


In [25]:
loss = binary_cross_entropy_loss(y_pred, y)
print(loss)

tensor(6.9010, grad_fn=<NegBackward0>)


In [26]:
loss.backward()

In [27]:
print(w.grad, b.grad)

tensor(6.8931) tensor(0.9990)


In [28]:
# till now we were performing operations on scaler input, we can do it using vector also

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = (x * x).mean()

print(y)

tensor(4.6667, grad_fn=<MeanBackward0>)


In [30]:
y.backward()
print(x.grad)

tensor([0.6667, 1.3333, 2.0000])


In [31]:
# if we repeatedly run forward-backward chain the gradient gets add on old gradient value, it does not gets clear automatically
# we need to clear gradient before repeat the forward-backward chain by making gradient zero using bellow function

x.grad.zero_()

tensor([0., 0., 0.])

In [32]:
# when we are training a neural network we will need gradient_track but after training is finished gradient will be no more required
# because now we will make predictions, and there will be no more backtracks needed, so we will stop this gradient track in bellow way
# there are 3 options to do this

# option 1: requires_grad(False) // this will make requires_grad=False
# option 2: detach() // it will create new tensor with no gradient
# option 3: torch.no_grad() // this will

# Tensor with gradient tracking enabled
x = torch.tensor([2.0], requires_grad=True)

# Standard operation tracks gradients
y = x ** 2
print(y.requires_grad)  # Output: True

# Operations inside the block do not track gradients
with torch.no_grad():
    z = x ** 2
    print(z.requires_grad)  # Output: False


True
False


In [33]:
# example for detach()
# Create a tensor that tracks gradients
x = torch.tensor([2.0], requires_grad=True)

# Normal operation tracks history
y = x ** 2
print(y.requires_grad)  # Output: True

# Detach y from the graph
z = y.detach()
print(z.requires_grad)  # Output: False

# Modifying z does not affect y's gradient history
print(z)

True
False
tensor([4.])
